# Descriptive tables and data quality

This notebook is the reproducible descriptive-data stage for the warm-up/cool-down experiment. It writes CSV tables to this directory.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'warmup_cooldown' / 'experiments').exists()), Path.cwd())
DATA = ROOT / 'warmup_cooldown' / 'experiments' / 'warmup_cooldown' / 'run_table.csv'
OUT = ROOT / 'warmup_cooldown' / 'statistics'
OUT.mkdir(parents=True, exist_ok=True)
DURATIONS = [5, 30, 60, 120, 210]
BENCHMARK_ORDER = ['n-body', 'spectral-norm', 'scimark-lu', 'scimark-sor', 'scimark-sparse', 'binary-trees']
raw = pd.read_csv(DATA)
required = {'__done', 'duration', 'benchmark', 'energy_j', 'benchmark_time_s', 'total_time_s', 'timestamp'}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')
df = raw.rename(columns={'duration':'duration_s', 'benchmark':'benchmark', 'energy_j':'energy_j', 'benchmark_time_s':'benchmark_time_s', 'total_time_s':'total_time_s'}).copy()
df['completed'] = df['__done'].eq('DONE')
for column in ['duration_s', 'energy_j', 'benchmark_time_s', 'total_time_s']:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
quality = pd.DataFrame([
    {'metric': 'raw_rows', 'value': len(raw)},
    {'metric': 'completed_rows', 'value': int(df['completed'].sum())},
    {'metric': 'not_done_rows', 'value': int((~df['completed']).sum())},
    {'metric': 'completed_rows_with_missing_or_invalid_values', 'value': int(df.loc[df['completed'], ['duration_s','energy_j','benchmark_time_s','total_time_s','timestamp']].isna().any(axis=1).sum())},
])
quality.to_csv(OUT / 'table_data_quality.csv', index=False)
df = df[df['completed']].dropna(subset=['duration_s','benchmark','energy_j','benchmark_time_s','total_time_s','timestamp']).copy()
df['duration_s'] = df['duration_s'].astype(int)
df = df[df['duration_s'].isin(DURATIONS) & df['energy_j'].ge(0) & df['benchmark_time_s'].ge(0) & df['total_time_s'].ge(0)].copy()
df['benchmark'] = pd.Categorical(df['benchmark'], categories=BENCHMARK_ORDER, ordered=True)
cell_median = df.groupby(['benchmark', 'duration_s'], observed=True)['energy_j'].transform('median')
df['absolute_deviation_j'] = (df['energy_j'] - cell_median).abs()
df.to_csv(OUT / 'cleaned_runs.csv', index=False)
print(f'Loaded {len(raw)} rows; retained {len(df)} valid completed rows.')

In [ ]:
grouped = df.groupby(['benchmark', 'duration_s'], observed=True)
summary = grouped.agg(
    n=('energy_j', 'size'),
    energy_mean_j=('energy_j', 'mean'),
    energy_median_j=('energy_j', 'median'),
    energy_sd_j=('energy_j', 'std'),
    energy_q1_j=('energy_j', lambda x: x.quantile(.25)),
    energy_q3_j=('energy_j', lambda x: x.quantile(.75)),
    benchmark_time_median_s=('benchmark_time_s', 'median'),
    total_time_median_s=('total_time_s', 'median'),
    total_time_mean_s=('total_time_s', 'mean'),
    mean_absolute_deviation_j=('absolute_deviation_j', 'mean'),
    sd_absolute_deviation_j=('absolute_deviation_j', 'std'),
).reset_index()
summary.to_csv(OUT / 'table_descriptive_by_cell.csv', index=False)
table4 = summary.pivot(index='benchmark', columns='duration_s')
table4.to_csv(OUT / 'table4_descriptive_wide.csv')
counts = df.groupby(['benchmark', 'duration_s'], observed=True).size().unstack(fill_value=0)
counts.to_csv(OUT / 'table_cell_counts.csv')
display(summary.round(3))

The primary RQ0 table is `table_descriptive_by_cell.csv`. The wide table is a convenient report-ready export; the long table is safer for plotting and further analysis.